<a href="https://colab.research.google.com/github/LizLian/from_scratch_2025/blob/main/bpe_tokenizer_20260317.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Step 1. Write a basic tokenizer class
class BasicTokenizer:
  def __init__(self):
    # default: vocab size of 256 (all bytes), no merges, no patterns
    self.merges = {} # (int, int) -> int
    self.pattern = "" # str
    self.special_tokens = {} # str -> int, e.g. {'<|endoftext|>': 100257}
    self.vocab = self._build_vocab() # int -> bytes

  def train(self, text, vocab_size, verbose=False):
    # Tokenizer can train a vocabulary of size vocab_size from text
    pass

  def encode(self, text):
    # Tokenizer can encode a string into a list of integers
    pass

  def decode(self, ids):
    # Tokenizer can decode a list of integers into a string
    pass

  def _build_vocab(self):
    # vocab is simply and deterministically derived from merges
    pass

In [ ]:
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode("utf-8") # raw bytes

tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience
print('---')
print(text)
print("length:", len(text))
print('---')
print(tokens)
print("length:", len(tokens))

---
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
length: 533
---
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140

In [ ]:
# @title Merge Function
# 1) get stats function that counts the # of times the pair appears
# 2) merge a pair to an ID in the given ids.

def get_stats(ids: list[int]):
  counts = {}
  for pair in zip(ids, ids[1:]):
    counts[pair] = counts.get(pair, 0) + 1
  return counts


def merge(ids, pair, idx):
  new_ids = []
  i = 0
  while i < len(ids):
    if i < len(ids) - 1 and pair[0] == ids[i] and pair[1] == ids[i+1]:
      new_ids.append(idx)
      i += 2
    else:
      new_ids.append(ids[i])
      i += 1
  return new_ids

In [ ]:
stats = get_stats(tokens)
print(sorted(((v, k) for k, v in stats.items()), reverse=True))

[(20, (101, 32)), (15, (240, 159)), (12, (226, 128)), (12, (105, 110)), (10, (115, 32)), (10, (97, 110)), (10, (32, 97)), (9, (32, 116)), (8, (116, 104)), (7, (159, 135)), (7, (159, 133)), (7, (97, 114)), (6, (239, 189)), (6, (140, 240)), (6, (128, 140)), (6, (116, 32)), (6, (114, 32)), (6, (111, 114)), (6, (110, 103)), (6, (110, 100)), (6, (109, 101)), (6, (104, 101)), (6, (101, 114)), (6, (32, 105)), (5, (117, 115)), (5, (115, 116)), (5, (110, 32)), (5, (100, 101)), (5, (44, 32)), (5, (32, 115)), (4, (116, 105)), (4, (116, 101)), (4, (115, 44)), (4, (114, 105)), (4, (111, 117)), (4, (111, 100)), (4, (110, 116)), (4, (110, 105)), (4, (105, 99)), (4, (104, 97)), (4, (103, 32)), (4, (101, 97)), (4, (100, 32)), (4, (99, 111)), (4, (97, 109)), (4, (85, 110)), (4, (32, 119)), (4, (32, 111)), (4, (32, 102)), (4, (32, 85)), (3, (118, 101)), (3, (116, 115)), (3, (116, 114)), (3, (116, 111)), (3, (114, 116)), (3, (114, 115)), (3, (114, 101)), (3, (111, 102)), (3, (111, 32)), (3, (108, 108)), (

In [ ]:
print(stats.get((101, 32)))

20


In [ ]:
top_pair = max(stats, key=stats.get)

In [ ]:
# @title Merge 20 times
from google3.pyglib import gfile

vocab_size = 330
num_merges = vocab_size - 256

with gfile.Open("/cns/wf-d/home/lizliang/dataset/from_scratch/taylorswift.txt") as f:
  text = f.read()

tokens = text.encode("utf-8")
ids = list(map(int, tokens))

merges = {}
for i in range(num_merges):
  stats = get_stats(ids)
  top_pair = max(stats, key=stats.get)
  idx = 256 + i
  ids = merge(ids, top_pair, idx)
  merges[top_pair] = idx
  print(f"merging {top_pair} into a new token {idx}")

merging (101, 32) into a new token 256
merging (44, 32) into a new token 257
merging (100, 32) into a new token 258
merging (46, 32) into a new token 259
merging (114, 32) into a new token 260
merging (50, 48) into a new token 261
merging (115, 32) into a new token 262
merging (105, 110) into a new token 263
merging (111, 110) into a new token 264
merging (114, 105) into a new token 265
merging (116, 32) into a new token 266
merging (116, 104) into a new token 267
merging (101, 258) into a new token 268
merging (257, 261) into a new token 269
merging (97, 110) into a new token 270
merging (97, 114) into a new token 271
merging (101, 260) into a new token 272
merging (121, 32) into a new token 273
merging (97, 108) into a new token 274
merging (267, 256) into a new token 275
merging (118, 268) into a new token 276
merging (119, 105) into a new token 277
merging (101, 114) into a new token 278
merging (264, 32) into a new token 279
merging (277, 102) into a new token 280
merging (82, 101

In [ ]:
# @title Build Vocab and the decode function
vocab = {i: bytes([i]) for i in range(256)}
for (p0, p1), idx in merges.items():
  vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
  tokens = b""
  for idx in ids:
    tokens += vocab[idx]
  text = tokens.decode("utf-8", errors="replace")
  return text

print(decode([0,3,4,90,12,51]))
print(vocab[0])

 Z3
b'\x00'


In [ ]:
# @title Build the encode function
def encode(text: str):
  """given a string, returns a list of integers."""
  tokens = list(map(int, text.encode("utf-8")))
  new_ids = []
  i = 0
  while i < len(tokens):
    if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) in merges:
      print((tokens[i], tokens[i+1]))
      new_ids.append(merges[(tokens[i], tokens[i+1])])
      i += 2
    else:
      new_ids.append(tokens[i])
      i += 1
  return new_ids



In [ ]:
encode("hello world!") == [104, 101, 108, 108, 111, 32, 119, 266, 108, 100, 33]

False

In [ ]:
encode("hello world!")

(108, 108)
(111, 114)


[104, 101, 301, 111, 32, 119, 291, 108, 100, 33]

In [ ]:
def encode_test(text):
  tokens = list(map(int, text.encode("utf-8")))
  while len(tokens) >= 2:
    stats = get_stats(tokens)
    pair = min(stats, key=lambda x: merges.get(x, float("inf")))
    if pair not in merges:
      break
    idx = merges[pair]
    tokens = merge(tokens, pair, idx)
  return tokens

In [ ]:
encode_test("hello world!")

[104, 101, 301, 111, 32, 119, 291, 108, 100, 33]